In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
name = torch.cuda.get_device_name(0); print("GPU:", name, flush=True)
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/train_biencoder.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for p in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(p)
    if not os.path.exists(dst): os.symlink(p, dst)
base = os.path.dirname(glob.glob("/kaggle/input/**/stage1__model.safetensors", recursive=True)[0])
os.makedirs("/kaggle/working/base", exist_ok=True)
for f in ("model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"):
    dst = f"/kaggle/working/base/{f}"
    if not os.path.exists(dst): os.symlink(f"{base}/stage1__{f}", dst)
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
sys.argv = ["train_biencoder", "--prepacked", "/kaggle/working/pack",
            "--base-model", "/kaggle/working/base",
            "--epochs", "1", "--batch-size", "64", "--max-length", "128",
            "--learning-rate", "2e-5", "--max-pairs", "400000",
            "--output", "/kaggle/working/bi_encoder"]
from src.train_biencoder import main
main()
